<a href="https://colab.research.google.com/github/joryhh/Capstone-Project--Building-Agentic-AI-Systems/blob/main/Capstone_Main_Workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip uninstall -y chromadb opentelemetry-api opentelemetry-sdk \
    opentelemetry-exporter-otlp opentelemetry-exporter-otlp-proto-grpc numpy -q

!pip install -qU langchain langchain-groq langgraph langgraph-supervisor pydantic "numpy<2.0.0"

import os
os.environ["CHROMA_SERVER_NO_TELEMETRY"] = "1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.0/147.0 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.9/248.9 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.1/565.1 kB 45.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jax 0.7.2

In [ ]:
# CELL 0b — API keys (Colab Secrets, never hardcoded)
from google.colab import userdata

# Required: free key at https://console.groq.com/keys
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# Required for §8 (LangSmith) — note the EXACT variable name.
# LANGSMITH_TRACING_V2 is not real and fails silently — this is LANGCHAIN_TRACING_V2.
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = userdata.get("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = "capstone-requirements-reviewer"

In [ ]:
# CELL 1 — Shared schemas (freeze this with Member B before anyone else builds on it)
from __future__ import annotations
from typing import Literal, Optional
from pydantic import BaseModel, Field


class Requirement(BaseModel):
    """A single parsed requirement, as produced by B's Requirements Parser Tool."""
    id: str = Field(description="Stable identifier, e.g. 'REQ-001'")
    text: str = Field(description="The requirement statement, verbatim or lightly normalized")
    category: Optional[str] = Field(
        default=None,
        description="Optional grouping, e.g. 'authentication', 'notifications'",
    )


class ReviewFinding(BaseModel):
    """One issue raised by any reviewer, against one requirement."""
    requirement_id: str = Field(description="ID of the requirement this finding refers to")
    issue_type: Literal["missing", "ambiguous", "conflicting", "untestable"] = Field(
        description=(
            "missing: a requirement that should exist but doesn't; "
            "ambiguous: vague/subjective wording; "
            "conflicting: contradicts another requirement; "
            "untestable: cannot be objectively verified as met"
        )
    )
    severity: Literal["low", "medium", "high", "critical"] = Field(
        description="How much this issue would hurt the project if left unresolved"
    )
    reason: str = Field(description="One or two sentences explaining the issue")
    suggested_change: str = Field(description="A concrete, measurable rewrite or addition")


class ApprovedChange(BaseModel):
    """A PO decision on one ReviewFinding, as emitted by D's HITL and consumed by E's Editor."""
    requirement_id: str
    action: Literal["replace", "add", "apply_po_edit", "leave_unchanged"] = Field(
        description=(
            "replace: swap the requirement text with suggested_change; "
            "add: this is a brand-new requirement being inserted; "
            "apply_po_edit: use the PO's own edited_text instead of the suggestion; "
            "leave_unchanged: reviewer's finding was rejected, requirement stays as-is"
        )
    )
    edited_text: Optional[str] = Field(
        default=None, description="PO's own wording, only set when action == 'apply_po_edit'"
    )
    notes: Optional[str] = Field(default=None, description="Free-text PO rationale, optional")


print("Schemas frozen:", Requirement.__name__, ReviewFinding.__name__, ApprovedChange.__name__)

Schemas frozen: Requirement ReviewFinding ApprovedChange


In [6]:
# CELL 2 — STUB reviewers. Replace each with B/C's real with_structured_output call
# before the final submission. These intentionally ignore most of their input;
# that is only acceptable because they are scaffolding, never submitted as-is.
from langgraph.func import task


@task
def stub_completeness_reviewer(project_description: str, requirements: list[Requirement]) -> list[ReviewFinding]:
    """STUB — real version compares project_description against requirements to find gaps."""
    print(f"  [stub] completeness_reviewer called with {len(requirements)} requirement(s)")
    return [
        ReviewFinding(
            requirement_id="REQ-000",
            issue_type="missing",
            severity="high",
            reason="[STUB] Placeholder gap — replace with B's real completeness check.",
            suggested_change="[STUB] Add the missing requirement B's reviewer will identify.",
        )
    ]


@task
def stub_ambiguity_reviewer(requirements: list[Requirement]) -> list[ReviewFinding]:
    """STUB — real version flags vague/subjective terms and proposes a measurable rewrite."""
    print(f"  [stub] ambiguity_reviewer called with {len(requirements)} requirement(s)")
    if not requirements:
        return []
    return [
        ReviewFinding(
            requirement_id=requirements[0].id,
            issue_type="ambiguous",
            severity="medium",
            reason="[STUB] Placeholder vague-term flag — replace with B's real ambiguity check.",
            suggested_change="[STUB] Rewrite with a measurable threshold.",
        )
    ]


@task
def stub_conflict_reviewer(requirements: list[Requirement]) -> list[ReviewFinding]:
    """STUB — real version (C) compares requirement pairs for contradictions."""
    print(f"  [stub] conflict_reviewer called with {len(requirements)} requirement(s)")
    if len(requirements) < 2:
        return []
    return [
        ReviewFinding(
            requirement_id=requirements[1].id,
            issue_type="conflicting",
            severity="critical",
            reason="[STUB] Placeholder contradiction — replace with C's real conflict check.",
            suggested_change="[STUB] Reconcile with the conflicting requirement.",
        )
    ]


@task
def stub_testability_reviewer(requirements: list[Requirement]) -> list[ReviewFinding]:
    """STUB — real version (C) flags requirements with no objective pass/fail criterion."""
    print(f"  [stub] testability_reviewer called with {len(requirements)} requirement(s)")
    if not requirements:
        return []
    return [
        ReviewFinding(
            requirement_id=requirements[-1].id,
            issue_type="untestable",
            severity="medium",
            reason="[STUB] Placeholder untestable flag — replace with C's real testability check.",
            suggested_change="[STUB] Add a measurable acceptance criterion.",
        )
    ]

In [11]:
# CELL 3 — Router decision schema + Supervisor-as-router task
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)


class RouterDecision(BaseModel):
    """The Supervisor's routing decision — which reviewers should run on this input."""
    run_completeness: bool = Field(
        description="True only if there is a project description to compare requirements against"
    )
    run_ambiguity: bool = Field(
        description="True if at least one requirement exists to check for vague wording"
    )
    run_conflict: bool = Field(
        description="True only if there are at least two requirements that could contradict each other"
    )
    run_testability: bool = Field(
        description="True if at least one requirement exists to check for measurable acceptance criteria"
    )
    reason: str = Field(description="One short sentence justifying which reviewers were chosen and why")


router_llm = llm.with_structured_output(RouterDecision)


@task
def supervisor_route(project_description: Optional[str], requirements: list[Requirement]) -> RouterDecision:
    """Supervisor-as-router: an LLM call with constrained output decides who runs.

    This is the routing primitive from the course's structured-routing lesson,
    NOT a keyword rule. The LLM sees the actual counts/content and decides;
    we only apply a deterministic override afterward for cases that are not
    a judgment call but a logical impossibility (see _enforce_hard_guardrails).
    """
    prompt = f"""You are the Supervisor for a requirements-review pipeline.
Decide which reviewers should run on this input. Do not run a reviewer whose
job is logically impossible given the input (e.g. you cannot check completeness
against a project description that does not exist).

CRITICAL INSTRUCTION: For the boolean routing fields (run_completeness, run_ambiguity, run_conflict, run_testability), you MUST output strict JSON boolean values (true or false), NOT strings ("true" or "false").

Project description: {project_description if project_description else "(none provided)"}
Number of requirements: {len(requirements)}
Requirements:
{chr(10).join(f"- [{r.id}] {r.text}" for r in requirements) if requirements else "(none)"}
"""
    decision = router_llm.invoke(prompt)
    return _enforce_hard_guardrails(decision, project_description, requirements)


def _enforce_hard_guardrails(
    decision: RouterDecision, project_description: Optional[str], requirements: list[Requirement]
) -> RouterDecision:
    """Deterministic floor UNDER the LLM's decision, for cases that are not a judgment
    call at all: e.g. it is not possible to run completeness review with zero
    description, or conflict review with fewer than two requirements to compare.
    This does not replace the LLM's routing — it only prevents it from attempting
    something structurally impossible, exactly like the `Literal` type constrains
    the value space in the structured-routing lesson.
    """
    if not project_description:
        decision.run_completeness = False
    if len(requirements) < 2:
        decision.run_conflict = False
    if not requirements:
        decision.run_ambiguity = False
        decision.run_testability = False
    return decision

In [8]:
# CELL 4 — Supervisor-as-synthesizer
_SEVERITY_RANK = {"critical": 0, "high": 1, "medium": 2, "low": 3}


@task
def supervisor_synthesize(finding_lists: list[list[ReviewFinding]]) -> list[ReviewFinding]:
    """Flatten all reviewers' outputs, dedupe by (requirement_id, issue_type) keeping the
    highest-severity duplicate, then sort by severity (critical first).
    """
    flattened: list[ReviewFinding] = [f for group in finding_lists for f in group]

    deduped: dict[tuple[str, str], ReviewFinding] = {}
    for finding in flattened:
        key = (finding.requirement_id, finding.issue_type)
        existing = deduped.get(key)
        if existing is None or _SEVERITY_RANK[finding.severity] < _SEVERITY_RANK[existing.severity]:
            deduped[key] = finding

    ordered = sorted(deduped.values(), key=lambda f: _SEVERITY_RANK[f.severity])
    return ordered


def render_report(findings: list[ReviewFinding]) -> str:
    """Human-readable rendering of the final, deduped, severity-ordered report."""
    if not findings:
        return "No findings — requirements document passed all checks that ran."
    lines = ["# Requirements Review Report", ""]
    for f in findings:
        lines.append(f"- **[{f.severity.upper()}] {f.issue_type}** — `{f.requirement_id}`")
        lines.append(f"  - Reason: {f.reason}")
        lines.append(f"  - Suggested change: {f.suggested_change}")
    return "\n".join(lines)

In [9]:
# CELL 5 — Full entrypoint: Orchestrator-Worker pattern via the Functional API
from langgraph.func import entrypoint
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()  # shared infra; D owns the long-term Store separately


@entrypoint(checkpointer=checkpointer)
def review_pipeline(inputs: dict) -> dict:
    """Orchestrator-Worker requirements-review pipeline.

    inputs = {
        "project_description": str | None,
        "requirements": list[dict],  # each dict must match the Requirement schema
    }
    """
    project_description = inputs.get("project_description")
    requirements = [Requirement(**r) for r in inputs.get("requirements", [])]

    # --- Orchestrator: decide who runs -----------------------------------
    decision = supervisor_route(project_description, requirements).result()
    print(f"Routing decision: {decision.model_dump()}")

    # --- Workers: call ONLY the reviewers the orchestrator selected -------
    # Futures are collected first (not .result()'d immediately) so genuinely
    # selected reviewers can run concurrently — this is what makes it
    # Orchestrator-Worker rather than a serial chain.
    futures = []
    if decision.run_completeness:
        futures.append(stub_completeness_reviewer(project_description, requirements))
    if decision.run_ambiguity:
        futures.append(stub_ambiguity_reviewer(requirements))
    if decision.run_conflict:
        futures.append(stub_conflict_reviewer(requirements))
    if decision.run_testability:
        futures.append(stub_testability_reviewer(requirements))

    finding_lists = [f.result() for f in futures]

    # --- Synthesizer: merge, dedupe, order --------------------------------
    final_findings = supervisor_synthesize(finding_lists).result()

    return {
        "routing_decision": decision.model_dump(),
        "findings": [f.model_dump() for f in final_findings],
        "report": render_report(final_findings),
    }

In [12]:
# CELL 6 — Run 1: full input (description + 3 requirements) — expect all 4 reviewers to run
config_a = {"configurable": {"thread_id": "demo-full-input"}}

full_input = {
    "project_description": (
        "A course-registration system for university students. Students should be able "
        "to register for courses, drop courses, and receive notifications about seat "
        "availability."
    ),
    "requirements": [
        {"id": "REQ-001", "text": "The system shall let a student register for a course."},
        {"id": "REQ-002", "text": "The system shall respond quickly to registration requests."},
        {"id": "REQ-003", "text": "The system shall never allow a student to drop a course after the add/drop deadline, "
                                   "except that the system shall always allow drops within 24 hours of registration."},
    ],
}

result_a = review_pipeline.invoke(full_input, config_a)
print("\n--- REPORT (full input) ---")
print(result_a["report"])

Routing decision: {'run_completeness': True, 'run_ambiguity': True, 'run_conflict': True, 'run_testability': True, 'reason': 'The project has a description and multiple requirements, so completeness, ambiguity, conflict, and testability checks are applicable.'}
  [stub] completeness_reviewer called with 3 requirement(s)
  [stub] conflict_reviewer called with 3 requirement(s)
  [stub] ambiguity_reviewer called with 3 requirement(s)
  [stub] testability_reviewer called with 3 requirement(s)

--- REPORT (full input) ---
# Requirements Review Report

- **[CRITICAL] conflicting** — `REQ-002`
  - Reason: [STUB] Placeholder contradiction — replace with C's real conflict check.
  - Suggested change: [STUB] Reconcile with the conflicting requirement.
- **[HIGH] missing** — `REQ-000`
  - Reason: [STUB] Placeholder gap — replace with B's real completeness check.
  - Suggested change: [STUB] Add the missing requirement B's reviewer will identify.
- **[MEDIUM] ambiguous** — `REQ-001`
  - Reason: [S

In [13]:
# CELL 7 — Run 2: no description, 1 requirement — expect completeness AND conflict to be skipped
config_b = {"configurable": {"thread_id": "demo-sparse-input"}}

sparse_input = {
    "project_description": None,
    "requirements": [
        {"id": "REQ-010", "text": "The system shall be user-friendly."},
    ],
}

result_b = review_pipeline.invoke(sparse_input, config_b)
print("\n--- REPORT (sparse input) ---")
print(result_b["report"])

# Direct proof of selectivity: compare the two routing decisions
print("\nRun 1 decision:", result_a["routing_decision"])
print("Run 2 decision:", result_b["routing_decision"])
assert result_a["routing_decision"]["run_completeness"] is True
assert result_b["routing_decision"]["run_completeness"] is False
assert result_b["routing_decision"]["run_conflict"] is False
print("\n✅ Selectivity confirmed: routing genuinely differs by input.")

Routing decision: {'run_completeness': False, 'run_ambiguity': True, 'run_conflict': False, 'run_testability': True, 'reason': 'Only one requirement exists with no project description to compare against'}
  [stub] ambiguity_reviewer called with 1 requirement(s)
  [stub] testability_reviewer called with 1 requirement(s)

--- REPORT (sparse input) ---
# Requirements Review Report

- **[MEDIUM] ambiguous** — `REQ-010`
  - Reason: [STUB] Placeholder vague-term flag — replace with B's real ambiguity check.
  - Suggested change: [STUB] Rewrite with a measurable threshold.
- **[MEDIUM] untestable** — `REQ-010`
  - Reason: [STUB] Placeholder untestable flag — replace with C's real testability check.
  - Suggested change: [STUB] Add a measurable acceptance criterion.

Run 1 decision: {'run_completeness': True, 'run_ambiguity': True, 'run_conflict': True, 'run_testability': True, 'reason': 'The project has a description and multiple requirements, so completeness, ambiguity, conflict, and testabi

In [14]:
# CELL 8 — Integration smoke test: routing + checkpointer + a real interrupt(),
# standing in for D's HITL node ahead of E's Editor. Replace `pending_editor_step`
# with D and E's real implementation once both land — this cell only proves the
# three pieces (routing, checkpointer, interrupt) coexist without breaking each other.
from langgraph.types import interrupt, Command


@task
def pending_editor_step(findings: list[ReviewFinding]) -> str:
    """STUB for D+E's real Approve/Edit/Reject flow. Pauses before any requirement
    text is actually changed — same placement the real Editor will use.
    """
    decision = interrupt({
        "action": "Approve, edit, or reject the proposed findings before they're applied",
        "findings": [f.model_dump() for f in findings],
    })
    return f"[STUB] Editor received PO decision: {decision}"


@entrypoint(checkpointer=checkpointer)
def review_pipeline_with_pause(inputs: dict) -> dict:
    project_description = inputs.get("project_description")
    requirements = [Requirement(**r) for r in inputs.get("requirements", [])]

    decision = supervisor_route(project_description, requirements).result()

    futures = []
    if decision.run_completeness:
        futures.append(stub_completeness_reviewer(project_description, requirements))
    if decision.run_ambiguity:
        futures.append(stub_ambiguity_reviewer(requirements))
    if decision.run_conflict:
        futures.append(stub_conflict_reviewer(requirements))
    if decision.run_testability:
        futures.append(stub_testability_reviewer(requirements))

    finding_lists = [f.result() for f in futures]
    final_findings = supervisor_synthesize(finding_lists).result()

    editor_result = pending_editor_step(final_findings).result()
    return {"routing_decision": decision.model_dump(), "editor_result": editor_result}


# --- Run it: pause, then resume ---
cfg = {"configurable": {"thread_id": "integration-check-1"}}

paused = review_pipeline_with_pause.invoke(full_input, cfg)
print("PAUSED — interrupt payload:", paused["__interrupt__"][0].value["action"])
print("Findings awaiting approval:", len(paused["__interrupt__"][0].value["findings"]))

resumed = review_pipeline_with_pause.invoke(Command(resume="approve"), cfg)
print("\nRESUMED — final result:", resumed)

  [stub] completeness_reviewer called with 3 requirement(s)
  [stub] ambiguity_reviewer called with 3 requirement(s)
  [stub] testability_reviewer called with 3 requirement(s)
  [stub] conflict_reviewer called with 3 requirement(s)
PAUSED — interrupt payload: Approve, edit, or reject the proposed findings before they're applied
Findings awaiting approval: 4

RESUMED — final result: {'routing_decision': {'run_completeness': True, 'run_ambiguity': True, 'run_conflict': True, 'run_testability': True, 'reason': 'The project has a description and multiple requirements, so completeness, ambiguity, conflict, and testability checks are applicable.'}, 'editor_result': '[STUB] Editor received PO decision: approve'}
